# 5 — Every figure in the paper

All eight figures, regenerated here from the stored results. Nothing in the
manuscript is drawn by hand; this notebook is the source of truth.

In [ ]:
#@title Install and import
%pip install -q qiskit qiskit-aer scikit-learn matplotlib pandas scipy

import json, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.metrics.pairwise import rbf_kernel, laplacian_kernel

sim = AerSimulator()
RES = "../results"
print("imports ok")

In [ ]:
#@title Shared style
plt.rcParams.update({"font.size": 9, "figure.dpi": 120,
                     "axes.grid": True, "grid.alpha": .22,
                     "axes.spines.top": False, "axes.spines.right": False})
C = {"q": "#1F4E79", "qp": "#5B9BD5", "c": "#C0504D", "k": "#7F7F7F",
     "s": "#9BBB59", "hl": "#E36C09"}
S = json.load(open(f"{RES}/scaling.json"))
FEATJ = json.load(open(f"{RES}/features.json"))
NS = sorted(int(k) for k in S["by_n"])
st = {n: S["by_n"][str(n)]["stats"] for n in NS}
print("loaded")

In [ ]:
#@title Figure 1 - which features carry the class signal
sep, order = FEATJ["separation"], FEATJ["order"]
fig, a = plt.subplots(figsize=(7, 2.6))
a.bar(range(len(order)), [sep[c] for c in order],
      color=[C["q"] if i < 12 else C["k"] for i in range(len(order))])
a.axvline(11.5, color=C["hl"], ls="--")
a.set_xticks(range(len(order)), order, rotation=90, fontsize=6)
a.set_ylabel("standardised mean difference"); a.set_xlabel("PCA component")
plt.tight_layout(); plt.show()

In [ ]:
#@title Figure 2 - held-out performance vs register size
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
x = np.arange(len(NS))
for key, col, lab, mk in [("quantum","q","quantum (entangled)","o-"),
                          ("product","qp","quantum (product)","s--"),
                          ("classical","c","classical kernel","^-"),
                          ("kmeans","k","k-means","v:"),
                          ("spectral","s","spectral","d:")]:
    ax[0].errorbar(x, [st[n][f"{key}_mean"] for n in NS],
                   yerr=[st[n][f"{key}_sd"] for n in NS], fmt=mk, color=C[col],
                   label=lab, ms=4, capsize=2.5, lw=1.2)
ax[0].set_xticks(x, [str(n) for n in NS]); ax[0].set_ylim(0, 1.15)
ax[0].set_xlabel("qubits n"); ax[0].set_ylabel("ARI (held out)")
ax[0].legend(fontsize=7, ncol=2, loc="lower left")

d = [st[n]["q_vs_c_diff"] for n in NS]
lo = [st[n]["q_vs_c_ci"][0] for n in NS]; hi = [st[n]["q_vs_c_ci"][1] for n in NS]
ax[1].axhline(0, color="k", lw=.8)
ax[1].errorbar(x, d, yerr=[np.array(d)-np.array(lo), np.array(hi)-np.array(d)],
               fmt="o-", color=C["q"], ms=4, capsize=3)
for i, n in enumerate(NS):
    ax[1].annotate(f"p={st[n]['q_vs_c_p']:.3f}", (x[i], hi[i]),
                   textcoords="offset points", xytext=(0,5), ha="center", fontsize=7)
ax[1].set_xticks(x, [str(n) for n in NS]); ax[1].set_xlabel("qubits n")
ax[1].set_ylabel("quantum - classical (ARI)")
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.show()
print("The sign changes with register size; no magnitude exceeds 0.013.")

In [ ]:
#@title Figure 3 - the search landscape
fams = ["angle","angle_ent","zz","iqp","reupload","hea","zy"]
short = {"angle":"ang","angle_ent":"ang+CX","zz":"ZZ","iqp":"IQP",
         "reupload":"re-up","hea":"HEA","zy":"ZY"}
fig, ax = plt.subplots(1, len(NS), figsize=(12, 3), sharey=True)
rng = np.random.default_rng(0)
for k, n in enumerate(NS):
    ds = S["by_n"][str(n)]["dev_scores"]
    for i, f in enumerate(fams):
        v = [r["dev"] for r in ds if r["family"] == f]
        ax[k].scatter(i + rng.uniform(-.24,.24,len(v)), v, s=6, alpha=.4, color=C["q"])
        ax[k].plot([i-.32, i+.32], [max(v)]*2, color=C["hl"], lw=1.8)
    ax[k].axhline(S["by_n"][str(n)]["classical"]["dev"], color=C["c"], ls="--")
    ax[k].set_xticks(range(len(fams)), [short[f] for f in fams], rotation=90, fontsize=7)
    ax[k].set_title(f"n={n}"); ax[k].grid(alpha=.25)
ax[0].set_ylabel("ARI (development)")
plt.tight_layout(); plt.show()
print("Spread WITHIN a family exceeds the spread between families.")

In [ ]:
#@title Figure 4 - bandwidth, and entanglement
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for n, col in zip(NS, [C["k"], C["s"], C["qp"], C["q"]]):
    ds = S["by_n"][str(n)]["dev_scores"]
    bw = sorted({r["bandwidth"] for r in ds})
    ax[0].plot(bw, [max(r["dev"] for r in ds if r["bandwidth"]==b) for b in bw],
               "o-", color=col, label=f"n={n}")
ax[0].set_xlabel("bandwidth c"); ax[0].set_ylabel("best ARI (development)")
ax[0].legend(fontsize=7); ax[0].set_ylim(0, 1)

w, xq = .34, np.arange(len(NS))
ax[1].bar(xq-w/2, [st[n]["product_mean"] for n in NS], w,
          yerr=[st[n]["product_sd"] for n in NS], color=C["qp"],
          capsize=2.5, label="product state")
ax[1].bar(xq+w/2, [st[n]["quantum_mean"] for n in NS], w,
          yerr=[st[n]["quantum_sd"] for n in NS], color=C["q"],
          capsize=2.5, label="entangled")
ax[1].set_xticks(xq, [str(n) for n in NS]); ax[1].set_xlabel("qubits n")
ax[1].set_ylabel("ARI (held out)"); ax[1].legend(fontsize=7, loc="lower center", ncol=2)
ax[1].set_ylim(0, 1.12)
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.show()

In [ ]:
#@title Figures 5 and 8 - budget ablation and kernel concentration
B = json.load(open(f"{RES}/budget.json"))
Cc = json.load(open(f"{RES}/concentration.json"))["extended"]
bs = B["budgets"]; ns = sorted(int(k) for k in Cc)

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
ax[0].axhline(0, color="k", lw=.8)
ax[0].errorbar(bs, [B["results"][str(b)]["margin_mean"] for b in bs],
               yerr=[B["results"][str(b)]["margin_sd"] for b in bs],
               fmt="o-", color=C["q"], capsize=3)
ax[0].set_xscale("log"); ax[0].set_xlabel("quantum search budget")
ax[0].set_ylabel("quantum - classical (ARI)"); ax[0].set_title("budget ablation")

ax[1].plot(ns, [Cc[str(n)]["gap"] for n in ns], "o-", color=C["hl"])
ax[1].set_xlabel("qubits n"); ax[1].set_ylabel("within - between class K")
ax[1].set_title("class signal in the kernel")

ax[2].plot(ns, [Cc[str(n)]["ari"] for n in ns], "o-", color=C["c"])
ax[2].set_xlabel("qubits n"); ax[2].set_ylabel("ARI"); ax[2].set_title("and performance")
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.show()

In [ ]:
#@title Figures 6 and 7 - class imbalance, and finite sampling on Aer
Bd = json.load(open(f"{RES}/baserate.json")); sw = Bd["sweep"]
H = pd.DataFrame(json.load(open(f"{RES}/shots.json"))["runs"])
pct = [r["pct"] for r in sw]

fig, ax = plt.subplots(1, 4, figsize=(15, 2.9))
for key, col, lab in [("q_ari","q","quantum"),("c_ari","c","classical"),("km_ari","k","k-means")]:
    ax[0].plot(pct, [r[key] for r in sw], "o-", color=C[col], label=lab)
ax[0].set_ylabel("ARI"); ax[0].set_title("clustering"); ax[0].legend(fontsize=7)
for key, col, lab in [("q_auc","q","quantum"),("c_auc","c","classical"),("dist_auc","k","centroid dist.")]:
    ax[1].plot(pct, [r[key] for r in sw], "o-", color=C[col], label=lab)
ax[1].set_ylabel("AUROC"); ax[1].set_title("anomaly scoring"); ax[1].legend(fontsize=7)
for a in ax[:2]:
    a.set_xscale("log"); a.set_xlabel("fraud share (%)")

for n, col, mk in [(8,"qp","s--"), (12,"q","o-")]:
    g = H[H.n_qubits == n].groupby("shots")
    ax[2].errorbar(sorted(H[H.n_qubits==n].shots.unique()), g.kernel_mae.mean(),
                   yerr=g.kernel_mae.std(), fmt=mk, color=C[col], capsize=2.5, label=f"n={n}")
    sub = H[H.n_qubits==n].copy(); sub["delta"] = sub.ari_measured - sub.ari_exact
    gg = sub.groupby("shots")
    ax[3].errorbar(sorted(sub.shots.unique()), gg.delta.mean(), yerr=gg.delta.std(),
                   fmt=mk, color=C[col], capsize=2.5, label=f"n={n}")
ax[2].set_xscale("log"); ax[2].set_yscale("log"); ax[2].set_ylabel("kernel MAE")
ax[3].axhline(0, color="k", lw=.8); ax[3].set_xscale("log")
ax[3].set_ylabel("ARI measured - exact"); ax[3].set_ylim(-.05,.05)
for a in ax[2:]:
    a.set_xlabel("shots"); a.legend(fontsize=7)
ax[2].set_title("kernel accuracy"); ax[3].set_title("clustering unaffected")
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.show()
print("Shot noise shrinks as 1/sqrt(N); the clustering does not move at all.")